# M3-B1 — Exploration des 3 sources Acerox

> Notebook d'**exploration rapide** — pas d'EDA fouillée, juste assez pour
> remplir l'inventaire de la note d'identification.

Auteur·rice : Nawelle — Date : 21/07/2026

**Règles** :
- Pas de transformation (juste lecture, `info`, `head`, `describe`)
- Une cellule markdown par source — qu'est-ce que tu observes ?
- Trace les **risques** et **questions** qui émergent pour l'`identification_sources.md`

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

DATA_DIR = Path("../data")

## Source 1 — Capteurs IoT (CSV)

In [2]:
df_iot = pd.read_csv(DATA_DIR / "capteurs_iot.csv")
df_iot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51000 entries, 0 to 50999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      51000 non-null  object 
 1   site           51000 non-null  object 
 2   line_id        51000 non-null  int64  
 3   sensor_id      51000 non-null  object 
 4   temperature_c  51000 non-null  float64
 5   vibration_mms  50251 non-null  float64
 6   debit_uh       51000 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 2.7+ MB


In [3]:
display(df_iot.describe())
display(df_iot.head())

print("Valeurs manquantes :")
display(df_iot.isna().sum())

print("Répartition par site :")
display(df_iot["site"].value_counts())

print("Répartition site x line_id :")
display(pd.crosstab(df_iot["site"], df_iot["line_id"]))

print("Doublons (timestamp, sensor_id) :", df_iot.duplicated(subset=["timestamp", "sensor_id"]).sum())

print("Lignes avec temperature_c > 100°C, par site/ligne :")
hi_temp = df_iot[df_iot["temperature_c"] > 100]
display(hi_temp["site"].value_counts())
display(hi_temp["line_id"].value_counts())

print("Vibration sur Roubaix LINE-3 : figée ou variable ?")
rl3 = df_iot[(df_iot["site"] == "Roubaix") & (df_iot["line_id"] == 3)]
display(rl3["vibration_mms"].value_counts().head())

print("Hypothèse unité : temperature_c de Roubaix LINE-3 convertie de °F en °C")
print("Moyenne brute :", rl3["temperature_c"].mean())
print("Moyenne convertie (x-32)*5/9 :", ((rl3["temperature_c"] - 32) * 5 / 9).mean())
print("Moyenne du reste du parc (hors Roubaix LINE-3) :",
      df_iot[~((df_iot["site"] == "Roubaix") & (df_iot["line_id"] == 3))]["temperature_c"].mean())

,line_id,temperature_c,vibration_mms,debit_uh
count,51000.000000,51000.000000,50251.000000,51000.000000
mean,2.005882,73.717034,4.831502,110.050188
std,1.033472,27.005577,2.685963,11.832749
min,1.000000,26.470000,0.000000,80.000000
25%,1.000000,60.237500,3.302777,102.000000
50%,2.000000,66.090000,4.187726,110.100000
75%,3.000000,72.840000,5.171534,118.050000
max,4.000000,160.000000,12.000000,150.000000


,timestamp,site,line_id,sensor_id,temperature_c,vibration_mms,debit_uh
0,2026-04-14T19:21:43,Lyon,1,SLYO-L1-T01,77.92,5.539793,101.27
1,2026-04-27T02:47:12,Lyon,1,SLYO-L1-T01,70.58,3.361715,110.19
2,2026-04-13T18:18:50,Saint-Etienne,1,SSAI-L1-T01,62.37,4.019277,111.28
3,2026-04-05T10:34:03,Roubaix,2,SROU-L2-T01,66.17,4.922531,123.93
4,2026-04-20T10:18:07,Saint-Etienne,3,SSAI-L3-T01,55.56,1.643043,101.40


Valeurs manquantes :


timestamp          0
site               0
line_id            0
sensor_id          0
temperature_c      0
vibration_mms    749
debit_uh           0
dtype: int64

Répartition par site :


...

site
Roubaix    5252
Name: count, dtype: int64

line_id
3    5252
Name: count, dtype: int64

Vibration sur Roubaix LINE-3 : figée ou variable ?


vibration_mms
12.0    5194
Name: count, dtype: int64

Hypothèse unité : temperature_c de Roubaix LINE-3 convertie de °F en °C
Moyenne brute : 149.9752779893374
Moyenne convertie (x-32)*5/9 : 65.54182110518744
Moyenne du reste du parc (hors Roubaix LINE-3) : 64.96237146979102


> **Observations** :
>
> - Volume : 51 000 relevés, 8 capteurs (1 par couple site/ligne), période 2026-04-01 → 2026-04-29
> - Période : couverture homogène sur tout le mois d'avril 2026
> - Qualité observée : 749 valeurs manquantes sur `vibration_mms` (1,5 %) ; 1 073 doublons sur `(timestamp, sensor_id)` ; couverture inégale des sites — Lyon n'a qu'une seule ligne suivie (LINE-1) contre 3 pour Saint-Étienne et 4 pour Roubaix
> - ⚠️ Deux anomalies distinctes sur **Roubaix LINE-3**, à ne pas confondre :
>   1. `vibration_mms` figée exactement à 12,0 mm/s sur 99 % des relevés (5 194/5 252) → ressemble à un capteur de vibration saturé/en panne (perte de signal réelle)
>   2. `temperature_c` > 100 °C (jusqu'à 160 °C, moyenne 150) — mais en convertissant ces valeurs de °F en °C (`(x-32)*5/9`), la moyenne retombe à **65,5 °C**, quasi identique à la moyenne du reste du parc (65,0 °C hors LINE-3) → ressemble beaucoup plus à un **bug d'unité** (capteur qui remonte en Fahrenheit sous une colonne nommée `temperature_c`) qu'à une vraie surchauffe. À vérifier avec l'équipe technique avant toute conclusion métier
> - Risques RGPD : aucune donnée à caractère personnel dans ce fichier (pas d'identifiant nominatif)
> - Pertinence métier : la vibration figée est probablement l'anomalie exploitable (vraie panne capteur) ; la "surchauffe" ne doit pas être prise pour argent comptant tant que l'unité n'est pas confirmée
> - Question pour Sébastien : le capteur de température de Roubaix LINE-3 est-il configuré en Fahrenheit ? Le capteur de vibration est-il connu comme défaillant (valeur de saturation de l'échelle) ?

## Source 2 — ERP (JSON)

In [4]:
with (DATA_DIR / "erp_export.json").open() as f:
    orders = json.load(f)
df_erp = pd.DataFrame(orders)
df_erp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ordre_id         2000 non-null   int64 
 1   produit_ref      2000 non-null   object
 2   site             2000 non-null   object
 3   line_id          2000 non-null   int64 
 4   date_lancement   2000 non-null   object
 5   date_fin_prevue  2000 non-null   object
 6   statut           2000 non-null   object
 7   ouvrier_id       1891 non-null   object
 8   quantite_kg      2000 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 140.8+ KB


In [5]:
display(df_erp.describe(include="all"))
display(df_erp.head())

print("Statuts :")
display(df_erp["statut"].value_counts())

print("Manquants ouvrier_id :", df_erp["ouvrier_id"].isna().sum())
print("Nombre d'employés distincts :", df_erp["ouvrier_id"].nunique())

print("Sites couverts :")
display(df_erp["site"].value_counts())

print("Répartition site x line_id :")
display(pd.crosstab(df_erp["site"], df_erp["line_id"]))

print("Période lancement :", df_erp["date_lancement"].min(), "→", df_erp["date_lancement"].max())

,ordre_id,produit_ref,site,line_id,date_lancement,date_fin_prevue,statut,ouvrier_id,quantite_kg
count,2000.000000,2000,2000,2000.000000,2000,2000,2000,1891,2000.000000
unique,NaN,10,2,NaN,1999,1999,4,1689,NaN
top,NaN,INOX-316-4,Roubaix,NaN,2026-04-29T19:55:27,2026-04-14T22:51:14,termine,EMP-4182,NaN
freq,NaN,227,1108,NaN,2,2,1559,4,NaN
mean,100999.500000,NaN,NaN,2.316000,NaN,NaN,NaN,NaN,2528.689000
std,577.494589,NaN,NaN,1.033769,NaN,NaN,NaN,NaN,1425.269007
min,100000.000000,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,51.000000
25%,100499.750000,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,1270.250000
50%,100999.500000,NaN,NaN,2.000000,NaN,NaN,NaN,NaN,2534.000000
75%,101499.250000,NaN,NaN,3.000000,NaN,NaN,NaN,NaN,3739.250000


,ordre_id,produit_ref,site,line_id,date_lancement,date_fin_prevue,statut,ouvrier_id,quantite_kg
0,100000,ALU-T1-22,Roubaix,3,2026-04-01T22:21:08,2026-04-02T23:21:08,suspendu,EMP-5317,3221
1,100001,INOX-316-4,Saint-Etienne,1,2026-04-26T14:52:52,2026-04-28T15:52:52,termine,EMP-7240,4556
2,100002,ALU-T2-18,Saint-Etienne,3,2026-04-11T09:54:06,2026-04-12T16:54:06,suspendu,EMP-1939,1308
3,100003,ALU-T1-22,Roubaix,1,2026-04-20T22:33:08,2026-04-22T04:33:08,termine,EMP-3531,2968
4,100004,ALU-T2-25,Roubaix,4,2026-04-24T01:03:02,2026-04-25T21:03:02,termine,EMP-8778,3278


Statuts :


statut
termine     1559
en_cours     197
suspendu     139
annule       105
Name: count, dtype: int64

Manquants ouvrier_id : 109
Nombre d'employés distincts : 1689
Sites couverts :


site
Roubaix          1108
Saint-Etienne     892
Name: count, dtype: int64

Répartition site x line_id :


line_id,1,2,3,4
site,,,,
Roubaix,264,265,276,303
Saint-Etienne,285,308,299,0


Période lancement : 2026-04-01T00:09:57 → 2026-04-29T23:20:33


> **Observations** :
>
> - Volume : 2 000 ordres de fabrication, 10 références produit, période 2026-04-01 → 2026-04-29
> - Statuts : `termine` (1 559), `en_cours` (197), `suspendu` (139), `annule` (105)
> - ⚠️ Couverture partielle : seuls **Roubaix** (1 108) et **Saint-Étienne** (892) sont présents — **aucun ordre pour Lyon**, alors que les capteurs IoT couvrent bien ce site → incohérence de périmètre entre les deux sources
> - ⚠️ Risque RGPD : `ouvrier_id` est un identifiant nominatif d'employé (format `EMP-XXXX`, ~1 689 employés distincts) — donnée à caractère personnel, à pseudonymiser/exclure de tout croisement avec des indicateurs de performance individuelle
> - Manquants : `ouvrier_id` absent sur 109 ordres (5,5 %) — à clarifier (saisie manquante ? ordre automatisé sans opérateur affecté ?)
> - Question pour Sébastien : pourquoi Lyon n'apparaît pas dans l'export ERP ? Un autre système gère-t-il ce site, ou est-ce un oubli d'export ?

## Source 3 — Logs machines (texte)

In [6]:
log_path = DATA_DIR / "logs_machines.log"
n_lines = sum(1 for _ in log_path.open())
print(f"Nombre de lignes : {n_lines:,}")
print(f"Taille fichier : {log_path.stat().st_size / 1024:.1f} Ko")

# Aperçu des 5 premières lignes
with log_path.open() as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.rstrip())

Nombre de lignes : 30,000
Taille fichier : 1872.8 Ko
[2026-04-01T00:00:16] Lyon LINE-1 INFO: shift_changed
[2026-04-01T00:01:07] Saint-Etienne LINE-2 INFO: operator_login
[2026-04-01T00:01:34] Saint-Etienne LINE-3 ERROR: vibration_overlimit sensor=SSAI-L3-T01
[2026-04-01T00:04:18] Roubaix LINE-4 INFO: maintenance_completed
[2026-04-01T00:04:35] Lyon LINE-1 INFO: tooling_loaded


In [7]:
import re
from collections import Counter

pattern = re.compile(r"\[(.*?)\] (\S+(?: \S+)?) (LINE-\d+) (\w+): (\w+)")

levels = Counter()
sites = Counter()
events = Counter()
by_site_line = Counter()
warn_error_by_site_line = Counter()

with log_path.open() as f:
    for line in f:
        m = pattern.match(line)
        if not m:
            continue
        ts, site, line_id, level, event = m.groups()
        levels[level] += 1
        sites[site] += 1
        events[event] += 1
        by_site_line[(site, line_id)] += 1
        if level in ("WARN", "ERROR"):
            warn_error_by_site_line[(site, line_id)] += 1

print("Niveaux :")
display(pd.Series(levels, name="count"))

print("Sites :")
display(pd.Series(sites, name="count"))

print("Top événements :")
display(pd.Series(dict(events.most_common(10)), name="count"))

print("Taux WARN/ERROR par site/ligne :")
taux = pd.DataFrame(
    [
        {"site": site, "line_id": line_id, "total": total, "warn_error": warn_error_by_site_line[(site, line_id)],
         "taux": warn_error_by_site_line[(site, line_id)] / total}
        for (site, line_id), total in sorted(by_site_line.items())
    ]
)
display(taux)

Niveaux :


INFO     22501
ERROR     1741
WARN      5758
Name: count, dtype: int64

Sites :


Lyon              5989
Saint-Etienne    12041
Roubaix          11970
Name: count, dtype: int64

Top événements :


operator_login                    4615
maintenance_completed             4501
machine_started                   4488
shift_changed                     4473
tooling_loaded                    4424
vibration_threshold_approached    1443
temperature_drift_detected        1442
lubricant_low                     1440
throughput_below_target           1433
communication_lost                 468
Name: count, dtype: int64

Taux WARN/ERROR par site/ligne :


,site,line_id,total,warn_error,taux
0,Lyon,LINE-1,5989,1328,0.221740
1,Roubaix,LINE-1,2997,649,0.216550
2,Roubaix,LINE-2,2973,650,0.218634
3,Roubaix,LINE-3,2978,1487,0.499328
4,Roubaix,LINE-4,3022,673,0.222700
5,Saint-Etienne,LINE-1,4026,887,0.220318
6,Saint-Etienne,LINE-2,4048,936,0.231225
7,Saint-Etienne,LINE-3,3967,889,0.224099


> **Observations** :
>
> - Format : texte semi-structuré, une ligne par événement : `[timestamp] site LINE-n NIVEAU: evenement [sensor=...]`
> - Volume : 30 000 lignes, 1,9 Mo, période équivalente aux autres sources (avril 2026)
> - Niveaux : INFO (22 501), WARN (5 758), ERROR (1 741)
> - Parsing nécessaire : regex sur `[timestamp] site LINE-n NIVEAU: evenement`, avec un paramètre optionnel `sensor=...` sur certains événements (ex. `vibration_overlimit`) à extraire séparément
> - ⚠️ Croisement avec les capteurs IoT (corrélation Roubaix line 3 ?) : le taux WARN/ERROR de Roubaix LINE-3 (~50 %) est bien 2x supérieur aux autres lignes (~22 %), et ça coïncide avec les deux anomalies capteurs relevées sur la même ligne — **mais attention à ne pas sur-interpréter**. Une partie de ces alertes (`temperature_critical`, `temperature_drift_detected`) est probablement déclenchée par le même bug d'unité Fahrenheit/Celsius plutôt que par un vrai risque thermique (le système d'alerte applique un seuil pensé pour du °C à des valeurs en °F). Le taux d'alerte plus élevé s'explique en revanche plus sûrement par le vrai capteur de vibration en panne (`vibration_overlimit`). Les logs ne "confirment" donc pas une surchauffe réelle — ils confirment surtout que l'instrumentation de cette ligne a un problème (unité + panne), ce qui reste un signal utile en soi
> - Question pour Sébastien : une intervention de maintenance a-t-elle eu lieu sur Roubaix LINE-3 en avril ? Le capteur de vibration a-t-il été remplacé/recalibré depuis ? Le capteur de température de cette ligne a-t-il un firmware/unité différent des autres ?

## Synthèse pour `identification_sources.md`

Remplis le tableau d'inventaire dans `../identification_sources.md` à
partir des observations ci-dessus.